# 05 · Modelo protagonista: ViT-B/16 con cabeza dual

**Camanchaca-Predict — G5 · Proyecto Aplicado 2026-2**

Fine-tuneamos **ViT-B/16** (preentrenado ImageNet-21k → 1k) con **cabeza
dual**: regresión de `log10(V)` **+** clasificación en 4 bandas de seguridad.

**¿Por qué ViT para la camanchaca?** La niebla real es heterogénea y su señal
de visibilidad está en **relaciones globales** entre regiones de la imagen
(gradiente hacia el horizonte, atenuación dependiente de la distancia,
parches de niebla). La auto-atención del ViT modela directamente dependencias
entre cualquier par de parches; una CNN solo las captura de forma indirecta
y local (receptividad creciente). ResNet-50 queda como baseline (notebook 04).

**Estrategia en 2 fases (encaja en Colab free / T4):**

| Fase | Qué se entrena | LR | Épocas |
|---|---|---|---|
| A — warmup | solo cabezas (tronco congelado) | 1e-3 | 2 |
| B — fine-tuning | todo el modelo (LR diferenciales) | tronco 1e-5 / cabezas 1e-4 | 6 |

Requiere: splits (notebook 03). GPU T4 (~60–90 min). Si hay OOM → batch 16.

In [ ]:
# Setup estándar del proyecto
import json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

!pip install -q timm
import timm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/camanchaca")
except ImportError:
    ROOT = Path("local_workspace")

DATA_DIR = ROOT / "data"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
FIG_DIR = ROOT / "figures"
for p in (CKPT_DIR, RESULTS_DIR, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
print(f"Dispositivo: {DEVICE}")

In [ ]:
# Datos: mismos splits, mismas transformaciones que el baseline
train_df = pd.read_csv(DATA_DIR / "splits" / "train.csv")
val_df = pd.read_csv(DATA_DIR / "splits" / "val.csv")
test_df = pd.read_csv(DATA_DIR / "splits" / "test.csv")
with open(DATA_DIR / "splits" / "class_weights.json") as f:
    stats = json.load(f)
MU, SIGMA = stats["mu"], stats["sigma"]
CLASS_WEIGHTS = torch.tensor(
    [stats["class_weights"][b] for b in
     ("critico", "alto_riesgo", "precaucion", "aceptable")],
    dtype=torch.float32).to(DEVICE)
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
train_tf = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class VisibilityDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = self.transform(Image.open(r["path"]).convert("RGB"))
        return (img, float(r["y_norm"]), int(r["band_idx"]), float(r["visibility_m"]))

BATCH = 32          # <-- bajar a 16 si aparece CUDA OOM
train_loader = DataLoader(VisibilityDataset(train_df, train_tf), batch_size=BATCH,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(VisibilityDataset(val_df, eval_tf), batch_size=BATCH,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(VisibilityDataset(test_df, eval_tf), batch_size=BATCH,
                         num_workers=2, pin_memory=True)
print(f"Loaders listos · batch={BATCH}")

In [ ]:
# Modelo ViT-B/16 con cabeza dual (idéntico a src/camanchaca/models/)
class ViTVisibility(nn.Module):
    def __init__(self, n_bands=4, pretrained=True):
        super().__init__()
        self.trunk = timm.create_model(
            "vit_base_patch16_224.augreg_in21k_ft_in1k",
            pretrained=pretrained, num_classes=0)
        d = self.trunk.num_features
        self.head_reg = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 1))
        self.head_cls = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, n_bands))
    def forward(self, x):
        f = self.trunk(x)
        return self.head_reg(f).squeeze(-1), self.head_cls(f)

model = ViTVisibility(pretrained=True).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"ViT-B/16 · {n_params/1e6:.1f} M parámetros")

In [ ]:
# Pérdida conjunta + métricas (mismas que el baseline para comparar justo)
LAM = 0.6   # peso de la regresión frente a la clasificación

def dual_loss(out_reg, out_cls, y_norm, band_idx):
    mse = F.mse_loss(out_reg, y_norm)
    ce = F.cross_entropy(out_cls, band_idx, weight=CLASS_WEIGHTS)
    return LAM * mse + (1 - LAM) * ce, mse, ce

BAND_EDGES = (50.0, 100.0, 200.0)
BAND_NAMES = ("critico", "alto_riesgo", "precaucion", "aceptable")

def to_meters(y_norm):
    return 10 ** (y_norm * SIGMA + MU)

def compute_metrics(v_true, v_pred):
    from sklearn.metrics import f1_score
    err = v_pred - v_true
    mae = float(np.abs(err).mean())
    rmse = float(np.sqrt((err ** 2).mean()))
    mape = float((np.abs(err) / np.clip(v_true, 1e-6, None)).mean() * 100)
    ss_res = float((err ** 2).sum())
    ss_tot = float(((v_true - v_true.mean()) ** 2).sum())
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    b_true = np.digitize(v_true, BAND_EDGES)
    b_pred = np.digitize(v_pred, BAND_EDGES)
    f1 = float(f1_score(b_true, b_pred, average="macro", zero_division=0))
    peligro = v_true < 100.0
    falso_seguro = float((v_pred[peligro] >= 100.0).mean()) if peligro.any() else 0.0
    return {"mae_m": mae, "rmse_m": rmse, "mape_pct": mape, "r2": r2,
            "f1_macro": f1, "false_safe_rate": falso_seguro}

def run_epoch(model, loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    sum_loss, preds, vtrues = 0.0, [], []
    with torch.set_grad_enabled(training):
        for x, y_norm, band, v_true in loader:
            x = x.to(DEVICE, non_blocking=True)
            y_norm = y_norm.to(DEVICE, non_blocking=True)
            band = band.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16,
                                enabled=training and scaler is not None):
                out_reg, out_cls = model(x)
                loss, mse, ce = dual_loss(out_reg.float(), out_cls.float(), y_norm, band)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            sum_loss += loss.item() * x.size(0)
            preds.append(out_reg.float().detach().cpu().numpy())
            vtrues.append(v_true.numpy())
    return (sum_loss / len(loader.dataset),
            to_meters(np.concatenate(preds)),
            np.concatenate(vtrues))

## Fase A — warmup de las cabezas (tronco congelado)

Evita que gradientes grandes de cabezas aleatorias destruyan los pesos
preentrenados del tronco. Es rápido (solo se actualizan ~0.2 M parámetros).

In [ ]:
# FASE A: congelamos el tronco y entrenamos solo las cabezas
for p in model.trunk.parameters():
    p.requires_grad = False

EPOCHS_A = 2
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-3)
scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")
history = []

t0 = time.time()
for epoch in range(1, EPOCHS_A + 1):
    tr_loss, _, _ = run_epoch(model, train_loader, optimizer, scaler)
    with torch.no_grad():
        va_loss, va_pred, va_true = run_epoch(model, val_loader)
    m = compute_metrics(va_true, va_pred)
    history.append({"fase": "A", "epoch": epoch, "train_loss": tr_loss,
                    "val_loss": va_loss, **m})
    print(f"[A{epoch}] train={tr_loss:.4f} val={va_loss:.4f} MAE={m['mae_m']:6.1f} m "
          f"F1={m['f1_macro']:.3f} FS={m['false_safe_rate']:.3f}")

## Fase B — fine-tuning completo con LR diferenciales

El tronco aprende con LR bajo (1e-5, preserva el preentrenamiento 21k) y las
cabezas con LR alto (1e-4). Early stopping sobre MAE de validación;
checkpoint por mejora en Drive (mitigación R4/R5).

In [ ]:
# FASE B: descongelamos todo y entrenamos con LR diferenciales
for p in model.trunk.parameters():
    p.requires_grad = True

EPOCHS_B = 6
PATIENCE = 3
optimizer = torch.optim.AdamW([
    {"params": model.trunk.parameters(), "lr": 1e-5},
    {"params": model.head_reg.parameters(), "lr": 1e-4},
    {"params": model.head_cls.parameters(), "lr": 1e-4},
], weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_B)

best_mae, best_epoch, no_improve = float("inf"), -1, 0
for epoch in range(1, EPOCHS_B + 1):
    tr_loss, _, _ = run_epoch(model, train_loader, optimizer, scaler)
    with torch.no_grad():
        va_loss, va_pred, va_true = run_epoch(model, val_loader)
    m = compute_metrics(va_true, va_pred)
    history.append({"fase": "B", "epoch": EPOCHS_A + epoch, "train_loss": tr_loss,
                    "val_loss": va_loss, **m})
    marker = ""
    if m["mae_m"] < best_mae:
        best_mae, best_epoch, no_improve = m["mae_m"], EPOCHS_A + epoch, 0
        torch.save({"state_dict": model.state_dict(), "mu": MU, "sigma": SIGMA,
                    "lam": LAM, "epoch": EPOCHS_A + epoch, "val_metrics": m},
                   CKPT_DIR / "vitb16_best.pt")
        marker = "  ← mejor (guardado)"
    else:
        no_improve += 1
    print(f"[B{epoch}] train={tr_loss:.4f} val={va_loss:.4f} MAE={m['mae_m']:6.1f} m "
          f"F1={m['f1_macro']:.3f} FS={m['false_safe_rate']:.3f}{marker}")
    scheduler.step()
    if no_improve >= PATIENCE:
        print(f"Early stopping en B{epoch} (mejor: época {best_epoch})")
        break
print(f"\\nTiempo total: {(time.time()-t0)/60:.1f} min · mejor época {best_epoch} "
      f"(val MAE {best_mae:.1f} m)")

In [ ]:
# Curvas de aprendizaje (fases A + B)
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(hist["epoch"], hist["train_loss"], "o-", label="train")
axes[0].plot(hist["epoch"], hist["val_loss"], "s-", label="val")
axes[0].axvline(EPOCHS_A + 0.5, color="gray", ls=":", lw=1)
axes[0].text(EPOCHS_A + 0.6, axes[0].get_ylim()[1]*0.95, "fase B", color="gray")
axes[0].set_xlabel("época"); axes[0].set_ylabel("pérdida dual"); axes[0].legend()
axes[1].plot(hist["epoch"], hist["mae_m"], "o-", color="#b2182b")
axes[1].set_xlabel("época"); axes[1].set_ylabel("MAE (m)")
fig.suptitle("ViT-B/16 — curvas de aprendizaje (fase A: warmup, fase B: fine-tuning)")
plt.savefig(FIG_DIR / "vitb16_curvas.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Evaluación FINAL en test con el mejor checkpoint
ckpt = torch.load(CKPT_DIR / "vitb16_best.pt", map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
with torch.no_grad():
    _, te_pred, te_true = run_epoch(model, test_loader)
vit_metrics = compute_metrics(te_true, te_pred)
vit_metrics.update({"model": "vit_b16_dual", "n_params_M": round(n_params/1e6, 1),
                    "best_epoch": best_epoch, "lam_reg": LAM,
                    "train_time_min": round((time.time()-t0)/60, 1)})
print(json.dumps(vit_metrics, indent=2))

In [ ]:
# Matriz de confusión + reporte por banda (cabeza de clasificación implícita)
from sklearn.metrics import classification_report, confusion_matrix

b_true = np.digitize(te_true, BAND_EDGES)
b_pred = np.digitize(te_pred, BAND_EDGES)
cm = confusion_matrix(b_true, b_pred, labels=[0, 1, 2, 3])
fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(BAND_NAMES, rotation=25, ha="right")
ax.set_yticklabels(BAND_NAMES)
ax.set_xlabel("Predicha"); ax.set_ylabel("Real"); ax.set_title("ViT-B/16 · test")
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.savefig(FIG_DIR / "vitb16_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
print(classification_report(b_true, b_pred, target_names=BAND_NAMES, zero_division=0))

In [ ]:
# Exportamos métricas y figuras (Drive + repo local)
with open(RESULTS_DIR / "vitb16_metrics.json", "w") as f:
    json.dump(vit_metrics, f, indent=2)

REPO_LOCAL = Path("/content/camanchaca-predict")
if REPO_LOCAL.exists():
    import shutil
    (REPO_LOCAL / "reports" / "figures").mkdir(parents=True, exist_ok=True)
    for fig in ("vitb16_curvas.png", "vitb16_confusion.png"):
        shutil.copy(FIG_DIR / fig, REPO_LOCAL / "reports" / "figures" / fig)
    (REPO_LOCAL / "reports" / "tables").mkdir(parents=True, exist_ok=True)
    shutil.copy(RESULTS_DIR / "vitb16_metrics.json",
                REPO_LOCAL / "reports" / "tables" / "vitb16_metrics.json")
print("Métricas y figuras exportadas ✅")

## Checklist de interpretación

- ¿El ViT supera al ResNet-50 en MAE y en falso-seguro? (ese es el argumento
  central del proyecto)
- ¿La fase A estabilizó el inicio del fine-tuning? (comparar con un run sin
  warmup sería un ablation interesante para la entrega)
- Sobreajuste: si train baja y val sube en fase B → subir weight decay o
  congelar más bloques (R8).

**Siguiente:** notebook 06 · eval_compare — comparación final ResNet vs ViT,
figuras para la presentación y test cualitativo O-HAZE.